In [ ]:
!pip install -q docling docling-core transformers huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 931.9/931.9 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.2/79.2 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.6/122.6 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/

In [ ]:
!pip install requests==2.32.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 4.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.34.2
    Uninstalling requests-2.34.2:
      Successfully uninstalled requests-2.34.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
docling-core 2.99.0 requires requests<3.0.0,>=2.34.2, but you have requests 2.32.4 which is incompatible.


In [ ]:
# pdf_path
from google.colab import files
uploaded = files.upload()
if uploaded:
    pdf_path = Path(list(uploaded.keys())[0])

Saving Form_10_K_for_Duolingo_INC.pdf to Form_10_K_for_Duolingo_INC (1).pdf


In [ ]:
# ============================================================
# extract_chunk.py - High-Performance PDF -> Docling -> Chunks
# ============================================================
import argparse
import hashlib
import json
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import torch
from transformers import AutoTokenizer

from docling.chunking import HybridChunker
from docling.datamodel.pipeline_options import (
    AcceleratorDevice,
    AcceleratorOptions,
    PdfPipelineOptions,
)
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling_core.transforms.chunker.hierarchical_chunker import (
    ChunkingDocSerializer,
    ChunkingSerializerProvider,
)
from docling_core.transforms.chunker.tokenizer.huggingface import (
    HuggingFaceTokenizer,
)
from docling_core.transforms.serializer.markdown import (
    MarkdownParams,
    MarkdownTableSerializer,
)

# Detect Google Colab Environment
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ============================================================
# CONFIGURATION
# ============================================================
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
MAX_TOKENS = 400
EMBED_MAX_TOKENS = 480
MIN_ALNUM_CHARS = 30

_SEPARATOR_ROW = re.compile(r"^\|?[\s:\-|]+\|?$")
_NUMERIC_CELL = re.compile(r"^[\s$€£(\-–—+]*\d[\d,.\s]*[%)]?\s*[a-zA-Z]{0,2}$")


# ============================================================
# DATA MODEL
# ============================================================
@
class DocumentChunk:
    chunk_id: str
    document_id: str
    content: str
    embedding_text: str
    chunk_type: str = "text"
    page_number: Optional[int] = None
    headings: List[str] = field(default_factory=list)
    source_file: Optional[str] = None
    metadata: Dict = field(default_factory=dict)


# ============================================================
# TABLE HELPERS
# ============================================================
def profile_table(text: str) -> Dict:
    """Extract structural headers and row labels from markdown tables."""
    lines = [
        line.strip()
        for line in text.splitlines()
        if line.startswith("|") and not _SEPARATOR_ROW.match(line)
    ]
    rows = [[c.strip() for c in line.strip("|").split("|")] for line in lines if line]

    if not rows:
        return {"columns": [], "row_labels": [], "meaningful": False}

    columns = [c for c in rows[0] if c and not _NUMERIC_CELL.match(c)][:20]
    row_labels = [
        r[0] for r in rows[1:] if r and r[0] and not _NUMERIC_CELL.match(r[0])
    ][:40]

    return {
        "columns": columns,
        "row_labels": row_labels,
        "meaningful": len(rows) >= 2 and (len(columns) > 0 or len(row_labels) > 0),
    }


class MarkdownTableSerializerProvider(ChunkingSerializerProvider):
    def get_serializer(self, doc):
        return ChunkingDocSerializer(
            doc=doc,
            table_serializer=MarkdownTableSerializer(),
            params=MarkdownParams(compact_tables=True),
        )


# ============================================================
# FAST CHUNKER CLASS
# ============================================================
class FastDoclingChunker:
    def __init__(
        self,
        embedding_model: str = EMBEDDING_MODEL,
        doc_title: Optional[str] = None,
    ):
        self.doc_title = doc_title
        self.hf_tokenizer = AutoTokenizer.from_pretrained(embedding_model)
        self.tokenizer = HuggingFaceTokenizer(
            tokenizer=self.hf_tokenizer, max_tokens=MAX_TOKENS
        )

        # Configure GPU pipeline & disable OCR for fast parsing on digital PDFs
        pipeline_options = PdfPipelineOptions()
        device_type = (
            AcceleratorDevice.CUDA
            if torch.cuda.is_available()
            else AcceleratorDevice.CPU
        )
        pipeline_options.accelerator_options = AcceleratorOptions(
            num_threads=2, device=device_type
        )
        pipeline_options.do_ocr = False  # Bypasses OCR overhead on digital 10-Ks

        self.chunker = HybridChunker(
            tokenizer=self.tokenizer,
            repeat_table_header=True,
            merge_peers=True,
            serializer_provider=MarkdownTableSerializerProvider(),
        )
        self.converter = DocumentConverter(
            format_options={"pdf": PdfFormatOption(pipeline_options=pipeline_options)}
        )

    def _truncate(self, text: str) -> str:
        ids = self.hf_tokenizer.encode(text, add_special_tokens=False)
        return (
            self.hf_tokenizer.decode(ids[:EMBED_MAX_TOKENS])
            if len(ids) > EMBED_MAX_TOKENS
            else text
        )

    def process_pdf(
        self, pdf_path: Path, doc_id: str
    ) -> Tuple[List[DocumentChunk], str]:
        print(f"[INFO] Converting '{pdf_path.name}' via Docling...")
        doc = self.converter.convert(source=pdf_path).document
        raw_chunks = list(self.chunker.chunk(dl_doc=doc))
        print(f"[INFO] Raw chunks generated: {len(raw_chunks)}")

        final_chunks: List[DocumentChunk] = []
        for idx, raw in enumerate(raw_chunks):
            text = (raw.text or "").strip()
            headings = list(raw.meta.headings or [])

            # Detect Chunk Type
            labels = [
                str(getattr(i, "label", "")).lower()
                for i in raw.meta.doc_items
                if getattr(i, "label", None)
            ]
            chunk_type = "table" if any("table" in l for l in labels) else "text"

            # Junk Check: Skip empty, whitespace, or tiny strings
            if not text or len([c for c in text if c.isalnum()]) < MIN_ALNUM_CHARS:
                continue

            profile = profile_table(text) if chunk_type == "table" else None
            if chunk_type == "table" and not profile["meaningful"]:
                continue

            # Extract Page Numbers
            pages = sorted(
                {
                    int(prov.page_no)
                    for item in raw.meta.doc_items
                    for prov in getattr(item, "prov", []) or []
                    if getattr(prov, "page_no", None)
                }
            )
            page_num = pages[0] if pages else None

            # Build Context-Rich Embedding Text
            emb_parts = []
            if self.doc_title:
                emb_parts.append(f"Doc: {self.doc_title}")
            if headings:
                emb_parts.append(f"Headings: {' > '.join(headings)}")
            if profile:
                if profile["columns"]:
                    emb_parts.append(f"Cols: {', '.join(profile['columns'])}")
                if profile["row_labels"]:
                    emb_parts.append(f"Rows: {'; '.join(profile['row_labels'])}")
            emb_parts.append(text)

            embedding_text = self._truncate("\n".join(emb_parts))
            chunk_id = hashlib.sha256(
                f"{doc_id}:{idx}:{text}".encode()
            ).hexdigest()[:16]

            final_chunks.append(
                DocumentChunk(
                    chunk_id=chunk_id,
                    document_id=doc_id,
                    content=text,
                    embedding_text=embedding_text,
                    chunk_type=chunk_type,
                    page_number=page_num,
                    headings=headings,
                    source_file=pdf_path.name,
                    metadata={
                        "page_numbers": pages,
                        "columns": profile["columns"] if profile else [],
                        "row_labels": profile["row_labels"] if profile else [],
                    },
                )
            )

        return final_chunks, doc.export_to_markdown()


# ============================================================
# MAIN EXECUTION
# ============================================================
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--pdf", default="Form_10_K_for_Duolingo_INC.pdf")
    parser.add_argument("--doc-id", default="duolingo_10k")
    args, _ = parser.parse_known_args()

    pdf_path = Path(args.pdf)

    # Colab File Loader
    if not pdf_path.exists() and IN_COLAB:
        print(f"[INFO] '{pdf_path.name}' not found. Please upload your PDF file:")
        from google.colab import files

        uploaded = files.upload()
        if uploaded:
            pdf_path = Path(list(uploaded.keys())[0])

    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF file not found at: {pdf_path.resolve()}")

    out_dir = Path("data/extracted")
    out_dir.mkdir(parents=True, exist_ok=True)

    chunker = FastDoclingChunker(doc_title=args.doc_id)
    chunks, markdown_text = chunker.process_pdf(pdf_path, args.doc_id)

    # Save Output Artifacts
    (out_dir / "docling_document.md").write_text(markdown_text, encoding="utf-8")

    serialized = [c.__dict__ for c in chunks]
    manifest_path = out_dir / "chunks_manifest.json"
    manifest_path.write_text(
        json.dumps(serialized, indent=2, ensure_ascii=False), encoding="utf-8"
    )

    print("\n" + "=" * 60)
    print(f"[SUCCESS] Extracted {len(chunks)} valid chunks.")
    print(f"[SUCCESS] Saved manifest to: {manifest_path.resolve()}")
    print("=" * 60)


if __name__ == "__main__":
    main()

[INFO] Converting 'Form_10_K_for_Duolingo_INC.pdf' via Docling...


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (515 > 512). Running this sequence through the model will result in indexing errors


[INFO] Raw chunks generated: 381

[SUCCESS] Extracted 372 valid chunks.
[SUCCESS] Saved manifest to: /content/data/extracted/chunks_manifest.json
